In [1]:
import pandas as pd
import os
import hashlib
import pickle

# Train

## Create id to hash

In [20]:
train = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/train_summary.csv")

In [5]:
def generate_sequential_hash_id(naccid, index):
    # Combine sequential numbering with short hash
    short_hash = hashlib.sha256(str(naccid).encode('utf-8')).hexdigest()[:4]
    return f"P{index:04d}_{short_hash}"

# Generate mapping
naccid_to_uid = {naccid: generate_sequential_hash_id(naccid, idx) 
                  for idx, naccid in enumerate(train['ID'].unique(), 1)}
train['HASH_ID'] = train['ID'].map(naccid_to_uid)

In [8]:
train.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary_prompt,visit_summary,COHORT,HASH_ID
0,NACC814828,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,"The subject is an 83-year-old right-handed, En...",NACC,P0001_abd4
1,NACC142849,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Based on the clinical data, identify the neuro...",A. No listed option is correct\nB. Alzheimer's...,B,Alzheimer's disease pathology (AD) and Lewy bo...,Neuropath,<|im_start|>user\nYou will receive patient dat...,The subject is an 80-year-old male who present...,NACC,P0002_2155
2,NACC698306,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Normal Cognition (NC)\nB. Dementia (DE),B,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The patient is an 81-year-old White female who...,NACC,P0003_f7a1
3,NACC517465,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,A,Mild Cognitive Impairment (MCI),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The subject is a 61-year-old right-handed male...,NACC,P0004_a0d3
4,NACC215680,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Using the information provided, identify the p...",A. Systemic and environmental factors includin...,B,Alzheimer's disease (AD),Primary etiology,<|im_start|>user\nYou will receive patient dat...,The subject is a 72-year-old male of Puerto Ri...,NACC,P0005_ac7b


In [ ]:
assert len(train['HASH_ID'].unique()) == len(train)

43064

In [ ]:
id_to_hashid_train = dict(zip(train['ID'], train['HASH_ID']))

import pickle
with open("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/id_to_hashid_train.pkl", "wb") as f:
    pickle.dump(id_to_hashid_train, f)


## Hash IDs of all nacc data

In [11]:
base_dir = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np"
dest_dir = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np_HASH"

In [ ]:
import os
import pandas as pd

def replace_id_with_hashid_and_save(base_dir, dest_dir, id_to_hashid):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.csv'):
                src_path = os.path.join(root, file)
                # Compute relative path to base_dir
                rel_path = os.path.relpath(src_path, base_dir)
                dest_path = os.path.join(dest_dir, rel_path)
                # Ensure destination directory exists
                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                # Load CSV
                print(f"Loading from {src_path}")
                df = pd.read_csv(src_path)
                df = df.drop(['diag_summary', 'patient_summary', 'visit_summary_prompt', 'COHORT'], axis=1)
                # Replace ID with HASH_ID, but keep column name as 'ID'
                if 'ID' in df.columns:
                    df['ID'] = df['ID'].map(id_to_hashid).fillna(df['ID'])
                    
                print(f"Saving to {dest_path}")
                # Save to destination
                if "HASH" not in dest_path:
                    raise ValueError
                df.to_csv(dest_path, index=False)

# Example usage:
replace_id_with_hashid_and_save(base_dir, dest_dir, id_to_hashid_train)


Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/train_summary.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np_HASH/train_summary.csv
Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/skywork_style/gs_8/all_correct.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np_HASH/skywork_style/gs_8/all_correct.csv
Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/skywork_style/gs_8/train_filtered.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np_HASH/skywork_style/gs

# Test

## Create id to hash

In [3]:
test = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv")

/scratch/ipykernel_1508718/3908899712.py:1: DtypeWarning: Columns (20,22,24,28,43,45,50,93,94,95,96,97,98,99,100,101,102,103,104,105,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,825,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv")


In [4]:
def generate_sequential_hash_id(naccid, index):
    # Combine sequential numbering with short hash
    short_hash = hashlib.sha256(str(naccid).encode('utf-8')).hexdigest()[:4]
    return f"P{index:04d}_{short_hash}"

# Generate mapping
naccid_to_uid = {naccid: generate_sequential_hash_id(naccid, idx) 
                  for idx, naccid in enumerate(test['NACCID'].unique(), 1)}
test['HASH_ID'] = test['NACCID'].map(naccid_to_uid)

In [5]:
test.head()

,NACCID,NACCVNUM,NACCADC,PACKET,FORMVER,VISITMO,VISITDAY,VISITYR,NACCAVST,NACCNVST,...,NACCDICO,sort_key,sort_year,patient_summary,diag_summary,VISITGAP,visit_summary_prompt,visit_summary,COHORT,HASH_ID
0,NACC772223,1,5310,I,2.0,1,29,2009,1,1,...,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",1,<|im_start|>user\nYou will receive patient dat...,"The subject is a 78-year-old White, right-hand...",NACC,P0001_8fde
1,NACC900564,5,8974,F,2.0,12,2,2009,5,5,...,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",2,<|im_start|>user\nYou will receive patient dat...,"The subject is an 89-year-old right-handed, En...",NACC,P0002_4653
2,NACC586854,4,8658,T,3.0,8,23,2016,4,2,...,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",0,<|im_start|>user\nYou will receive patient dat...,The subject is a 63-year-old female who visite...,NACC,P0003_1e4e
3,NACC474857,3,2096,F,2.0,3,23,2010,3,3,...,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",2,<|im_start|>user\nYou will receive patient dat...,The subject is a 67-year-old male of White rac...,NACC,P0004_40a6
4,NACC316274,9,5783,F,2.0,10,1,2014,9,9,...,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",1,<|im_start|>user\nYou will receive patient dat...,"The subject is a 103-year-old right-handed, En...",NACC,P0005_ae2c


In [6]:
assert len(test['HASH_ID'].unique()) == len(test)

In [8]:
id_to_hashid_test = dict(zip(test['NACCID'], test['HASH_ID']))

import pickle
with open("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/id_to_hashid_test.pkl", "wb") as f:
    pickle.dump(id_to_hashid_test, f)


## Hash IDs of all nacc data

In [10]:
base_dir = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary"
dest_dir = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary_HASH"

In [12]:
import os
import pandas as pd

def replace_id_with_hashid_and_save(base_dir, dest_dir, id_to_hashid):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.csv'):
                src_path = os.path.join(root, file)
                # Compute relative path to base_dir
                rel_path = os.path.relpath(src_path, base_dir)
                dest_path = os.path.join(dest_dir, rel_path)
                # Ensure destination directory exists
                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                # Load CSV
                print(f"Loading from {src_path}")
                df = pd.read_csv(src_path)
                df = df.drop(['diag_summary', 'patient_summary', 'visit_summary_prompt', 'COHORT'], axis=1)
                # Replace ID with HASH_ID, but keep column name as 'ID'
                if 'ID' in df.columns:
                    df['ID'] = df['ID'].map(id_to_hashid).fillna(df['ID'])
                    
                print(f"Saving to {dest_path}")
                # Save to destination
                if "HASH" not in dest_path:
                    raise ValueError
                df.to_csv(dest_path, index=False)

# Example usage:
replace_id_with_hashid_and_save(base_dir, dest_dir, id_to_hashid_test)


Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_cog.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary_HASH/test_cog.csv
Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_np.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary_HASH/test_np.csv
Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_pet.csv
Saving to /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary_HASH/test_pet.csv
Loading from /projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/test

In [13]:
import pickle

with open("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/id_to_hashid_train.pkl", "rb") as f:
    id_to_hashid_train = pickle.load(f)


In [20]:
len(set(id_to_hashid_train.values()).intersection(set(id_to_hashid_test.values())))

0

# Convert HASH to ID

In [22]:
nacc_train = pd.read_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/train_with_questions.csv')

/scratch/ipykernel_212064/1288251784.py:1: DtypeWarning: Columns (13,40,56,62,70,86,88,109,138,143,146,149,190,246,247,248,250,251,252,253,254,255,256,257,258,259,261,262,263,264,265,266,267,268,269,298,334,336,339,340,342,351,454,501,503,511,519,521,535,537,538,590,599,655,660,682,738,743,746,804,812,813,817,818,831,869,873,875,877,879,899,900,907,908,959,960,961,962,966,967,987,1000,1001,1016,1020,1021,1023,1036,1041,1044,1050,1054,1055,1059,1066,1070,1071,1073,1090,1200,1218,1219,1232,1233,1234,1235,1236,1237,1241,1243,1247,1251,1255,1264,1266,1272,1273,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1294,1296,1381,1397,1423,1475,1486,1594,1595,1596,1600,1625,1654,1655,1656,1657,1660,1690,1754,1786,1788,1793,1813,1816,1820,1825,1827,1867,1871,1876,1890,1892,1893,1914,1939,1941,1943,1945,1947,2011,2025,2033,2045,2047,2069,2071,2083,2085,2087,2089,2091) have mixed types. Specify dtype option on import or set low_memory=False.
  nacc_train = pd.read_csv('/projectnb/vkolagrp/skow

In [46]:
with open("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/id_to_hashid_train.pkl", "rb") as f:
    id_to_hashid_train = pickle.load(f)

In [47]:
hashid_to_id_train = {v:k for k,v in id_to_hashid_train.items()}

In [48]:
df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np_HASH/subset/all_train_inc_oversample_sub.csv")

In [49]:
df.head()

,ID,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary
0,P12531_207f,Which of the following pathologies are causing...,A. Lewy body pathology (LBD) and Frontotempora...,A,Lewy body pathology (LBD) and Frontotemporal L...,Neuropath,The patient is an 87-year-old male who is Whit...
1,P24967_a5a5,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Mild Cognitive Impairment...,A,Dementia (DE),Cognitive status,The subject is a 75-year-old White female who ...
2,P19410_e169,Analyze the patient's information and identify...,A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,The patient is a 73-year-old white female who ...
3,P16002_d21a,"Using the information provided, identify the p...",A. Lewy body disease (LBD)\nB. Alzheimer's dis...,C,Frontotemporal lobar degeneration and its vari...,Primary etiology,The subject is a 64-year-old female who visite...
4,P30686_7dde,Identify the primary etiology of the patient's...,A. Alzheimer's disease (AD)\nB. Vascular brain...,D,Psychiatric conditions including schizophrenia...,Primary etiology,"The patient is an 88-year-old right-handed, En..."


In [50]:
hashid_to_id_train['P19410_e169']

'NACC018688'

In [51]:
df['ID'] = df['ID'].map(hashid_to_id_train)


In [52]:
df.head()

,ID,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary
0,NACC792378,Which of the following pathologies are causing...,A. Lewy body pathology (LBD) and Frontotempora...,A,Lewy body pathology (LBD) and Frontotemporal L...,Neuropath,The patient is an 87-year-old male who is Whit...
1,NACC322146,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Mild Cognitive Impairment...,A,Dementia (DE),Cognitive status,The subject is a 75-year-old White female who ...
2,NACC018688,Analyze the patient's information and identify...,A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,The patient is a 73-year-old white female who ...
3,NACC926277,"Using the information provided, identify the p...",A. Lewy body disease (LBD)\nB. Alzheimer's dis...,C,Frontotemporal lobar degeneration and its vari...,Primary etiology,The subject is a 64-year-old female who visite...
4,NACC211966,Identify the primary etiology of the patient's...,A. Alzheimer's disease (AD)\nB. Vascular brain...,D,Psychiatric conditions including schizophrenia...,Primary etiology,"The patient is an 88-year-old right-handed, En..."


In [53]:
df.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/subset/all_train_inc_oversample_sub.csv", index=False)

In [54]:
len(df)

23864

In [55]:
len(df['ID'].unique())

17847

In [56]:
df_deduplicated = df[df.duplicated() == False]


In [57]:
df_deduplicated.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/subset/all_train_inc_oversample_sub_deduplicated.csv", index=False)

In [58]:
nacc_train_sub = nacc_train[nacc_train['ID'].isin(df_deduplicated['ID'])].reset_index(drop=True)

In [59]:
nacc_train_sub.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary_prompt,visit_summary,...,VNTPCNC,VNTTOTW,WAIS,WEIGHT,WHITEVOL,WMHVOL,WONDRFUL,WRTHLESS,sort_key,sort_year
0,NACC814828,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,"The subject is an 83-year-old right-handed, En...",...,NaN,NaN,NaN,114.0,NaN,NaN,0.0,0.0,0,NaN
1,NACC142849,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Based on the clinical data, identify the neuro...",A. No listed option is correct\nB. Alzheimer's...,B,Alzheimer's disease pathology (AD) and Lewy bo...,Neuropath,<|im_start|>user\nYou will receive patient dat...,The subject is an 80-year-old male who present...,...,NaN,NaN,NaN,205.0,NaN,NaN,0.0,0.0,0,NaN
2,NACC517465,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,A,Mild Cognitive Impairment (MCI),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The subject is a 61-year-old right-handed male...,...,NaN,NaN,49.0,175.0,NaN,NaN,0.0,1.0,0,NaN
3,NACC734580,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Using the information provided, identify the p...",A. Not applicable (no cognitive impairment)\nB...,E,Lewy body disease (LBD),Primary etiology,<|im_start|>user\nYou will receive patient dat...,"The subject is a 64-year-old right-handed, Whi...",...,NaN,NaN,NaN,138.0,NaN,NaN,0.0,0.0,0,NaN
4,NACC924274,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Mild Cognitive Impairment (MCI)\nB. Dementi...,A,Mild Cognitive Impairment (MCI),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The subject is an 89-year-old female who is pa...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN


In [60]:
columns = set(df_deduplicated.columns).intersection(nacc_train_sub.columns)
columns

{'ID',
 'Q_TYPE',
 'ground_truth',
 'ground_truth_text',
 'options',
 'question',
 'visit_summary'}

In [61]:
merged_df = pd.merge(df_deduplicated, nacc_train_sub, on=list(columns), how='inner').reset_index(drop=True)


In [62]:
len(merged_df['ID'].unique())

17847

In [63]:
len(merged_df)

17847

In [64]:
merged_df.head()

,ID,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary,patient_summary,diag_summary,visit_summary_prompt,...,VNTPCNC,VNTTOTW,WAIS,WEIGHT,WHITEVOL,WMHVOL,WONDRFUL,WRTHLESS,sort_key,sort_year
0,NACC792378,Which of the following pathologies are causing...,A. Lewy body pathology (LBD) and Frontotempora...,A,Lewy body pathology (LBD) and Frontotemporal L...,Neuropath,The patient is an 87-year-old male who is Whit...,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",<|im_start|>user\nYou will receive patient dat...,...,NaN,NaN,17.0,171.0,NaN,NaN,0.0,0.0,0,NaN
1,NACC322146,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Mild Cognitive Impairment...,A,Dementia (DE),Cognitive status,The subject is a 75-year-old White female who ...,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",<|im_start|>user\nYou will receive patient dat...,...,NaN,NaN,NaN,145.0,NaN,NaN,0.0,0.0,0,NaN
2,NACC018688,Analyze the patient's information and identify...,A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,The patient is a 73-year-old white female who ...,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",<|im_start|>user\nYou will receive patient dat...,...,NaN,NaN,30.0,161.0,NaN,NaN,0.0,0.0,0,NaN
3,NACC926277,"Using the information provided, identify the p...",A. Lewy body disease (LBD)\nB. Alzheimer's dis...,C,Frontotemporal lobar degeneration and its vari...,Primary etiology,The subject is a 64-year-old female who visite...,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",<|im_start|>user\nYou will receive patient dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,NACC211966,Identify the primary etiology of the patient's...,A. Alzheimer's disease (AD)\nB. Vascular brain...,D,Psychiatric conditions including schizophrenia...,Primary etiology,"The patient is an 88-year-old right-handed, En...","{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",<|im_start|>user\nYou will receive patient dat...,...,NaN,NaN,97.0,162.0,NaN,NaN,0.0,0.0,0,NaN


In [65]:
merged_df.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo_modified_cog_np/subset/all_train_inc_oversample_sub_deduplicated_merged.csv", index=False)